# FZ0-Deep-Risk (Contribution 1) - Germany A1 run

End-to-end model that learns the (VaR, ES) pair directly under the Fissler-Ziegel
(FZ0) loss, run incrementally: **(1) data, (2) smoke test, (3) coverage check,
(4) full multi-seed, (5) test backtest vs. the two-stage pipeline**.

Reuses `pipeline_v2p.preprocess` (data) and `var_v2p` (backtest); the model, the
FZ0 loss and the training loop live in `fz0_deep_risk.py`. Requires `torch`.
Run the cells top to bottom and stop if the smoke test diverges.

## 1. Configuration and data

In [1]:
import importlib, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from darts import TimeSeries
import pipeline_v2p as pv2p, var_v2p as vv, fz0_deep_risk as fz
importlib.reload(fz); importlib.reload(vv)

CC, TARGET, H, INPUT_LEN = "GER", "ger_bmk", 5, 90
DATASET = "dataset/ds_steel.xlsx"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

d = pv2p.preprocess(TARGET, DATASET)
w = fz.make_windows(d, input_len=INPUT_LEN, h=H)
print("device:", DEVICE, "| n_cov:", w["n_cov"],
      "| train/val/test:", len(w["Xtr"]), len(w["Xva"]), len(w["Xte"]))

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


device: cuda | n_cov: 6 | train/val/test: 3158 751 1002


## 2. Smoke test (1 seed, few epochs)

Check that the validation FZ0 **decreases** and does not go NaN/explode, and that
the warmup->FZ0 switch (epoch 15) is smooth. If it diverges, lower `lr` to 5e-4.

In [2]:
torch.manual_seed(42); np.random.seed(42)
model = fz.FZ0DLinear(INPUT_LEN, n_cov=w["n_cov"])
model, valfz = fz.fit(model, w, epochs=60, warmup=15, patience=20, lr=1e-3,
                      device=DEVICE, verbose=True)
print("smoke val_FZ0:", round(valfz, 4))

  ep   0 | val=5.8305 (fz=4.386 pin=0.1445 med_bal=0.71) | best=5.8305
  ep  20 | val=-1.0517 (fz=-1.347 pin=0.0295 med_bal=0.50) | best=-1.1100
smoke val_FZ0: -1.11


## 3. Coverage sanity check

Transport (m, VaR) to price space (quantiles are monotone-transform equivariant)
and measure empirical 95% coverage. Target ~0.95 (acceptable 0.90-0.97).

In [3]:
def to_price(vals):
    ts = TimeSeries.from_times_and_values(w["test_time"], vals.reshape(-1, 1), columns=[TARGET])
    return pv2p.inverse_chain(ts, d["scaler"]).univariate_values()

m, v, e = fz.predict(model, w, device=DEVICE)
yhat = to_price(m); q95 = to_price(m + v)
sra = dict(zip(d["sra"].time_index, d["sra"].univariate_values()))
y = np.array([sra[t] for t in w["test_time"]])
L = np.log(y / yhat); VaR = np.log(q95 / yhat)
cov = 1 - (L > VaR).mean()
print(f"empirical 95% coverage: {cov:.3f}  (target ~0.95)")
print(f"exceedances: {int((L>VaR).sum())} of {len(L)} (expected ~{0.05*len(L):.0f})")

empirical 95% coverage: 0.951  (target ~0.95)
exceedances: 49 of 1002 (expected ~50)


## 4. Full multi-seed training

Only after the smoke test and coverage look sane. Median seed by validation FZ0.

In [4]:
runs = []
for s in [42, 43, 44]:
    torch.manual_seed(s); np.random.seed(s)
    mdl = fz.FZ0DLinear(INPUT_LEN, n_cov=w["n_cov"])
    mdl, vf = fz.fit(mdl, w, epochs=300, warmup=20, patience=25, lr=1e-3,
                     device=DEVICE, verbose=False)
    runs.append((s, vf, mdl)); print(f"seed {s}: val_FZ0={vf:.4f}")
runs.sort(key=lambda r: r[1])
seed_med, val_med, model = runs[len(runs)//2]
print("median seed:", seed_med, "| val_FZ0:", round(val_med, 4))

seed 42: val_FZ0=-1.1679
seed 43: val_FZ0=-1.2132
seed 44: val_FZ0=-1.2147
median seed: 43 | val_FZ0: -1.2132


## 5. Test backtest and comparison vs. the two-stage pipeline

The end-to-end model emits native (VaR, ES); we backtest them at 95% (Kupiec,
Christoffersen/DQ de-overlapped, FZ0) with `var_v2p`, and compare the FZ0 loss
series against the best two-stage engine for Germany via the DM test.

In [5]:
# end-to-end (VaR, ES) in price -> loss space, aligned to test index
m, v, e = fz.predict(model, w, device=DEVICE)
yhat = to_price(m); q95 = to_price(m + v); es = to_price(m + e)
y = np.array([sra[t] for t in w["test_time"]])
L_e = np.log(y / yhat)
V_e = np.log(q95 / yhat)
E_e = np.maximum(np.log(es / yhat), V_e)          # ES >= VaR after transport

hits = (L_e > V_e).astype(int)
fz0_e2e = float(np.nanmean(vv.fz0_loss(L_e, np.maximum(V_e,1e-4), E_e, vv.A)))
print("=== FZ0-DLinear (end-to-end), Germany ===")
print(f"FZ0={fz0_e2e:.4f} | coverage={1-hits.mean():.3f} | Kupiec={vv.kupiec(hits,vv.A):.3f} "
      f"| DQ_str={vv.dq_stride(hits, V_e, H):.3f} | exc={int(hits.sum())}")

# baseline: best two-stage engine FZ0 for GER (from the risk-layer results)
# TCN via GJR-GARCH-t was the GER champion; recompute its FZ0 series for the DM test.
RF = f"result/v2p_result_steel_{CC.lower()}.xlsx"
_, L_b, _ = vv.load_losses(RF, CC, H, "tcn")
V_b, E_b = vv.rolling_var_garch(L_b, vv.A, "GJR")
W = vv.WINDOW
fz_b = vv.fz0_loss(L_b[W:], np.maximum(V_b[W:],1e-4), np.maximum(E_b[W:],V_b[W:]), vv.A)
fz_e = vv.fz0_loss(L_e, np.maximum(V_e,1e-4), E_e, vv.A)
n = min(len(fz_b), len(fz_e))
stat, p = vv.dm_loss(fz_e[:n], fz_b[:n], nw_lag=H)   # p<0.05 => end-to-end better
print(f"\nDM  end-to-end vs TCN/GJR-GARCH (2-stage): dFZ0={np.nanmean(fz_e[:n])-np.nanmean(fz_b[:n]):+.4f} "
      f"| p={p:.3f}  ({'end-to-end better' if p<0.05 else 'not significant'})")

=== FZ0-DLinear (end-to-end), Germany ===
FZ0=-3.7923 | coverage=0.920 | Kupiec=0.000 | DQ_str=0.001 | exc=80

DM  end-to-end vs TCN/GJR-GARCH (2-stage): dFZ0=+0.8525 | p=0.999  (not significant)


## 6. Optuna

In [7]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    il  = trial.suggest_int("input_len", 60, 240, step=30)
    ker = trial.suggest_int("kernel", 5, 51, step=2)
    lr  = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    wloc = fz.make_windows(d, input_len=il, h=H)
    torch.manual_seed(42); np.random.seed(42)
    mdl = fz.FZ0DLinear(il, n_cov=wloc["n_cov"], kernel=ker)
    mdl, vf = fz.fit(mdl, wloc, epochs=200, warmup=30, patience=25, lr=lr,
                     device=DEVICE, verbose=False)
    # portão de validade DENTRO do objetivo: cobertura de validação
    import torch as _t
    with _t.no_grad():
        mv, vv_, _ = mdl(_t.tensor(wloc["Xva"], device=DEVICE),
                         _t.tensor(wloc["Cva"], device=DEVICE))
    cov = float(np.mean((wloc["Zva"] - mv.cpu().numpy()) <= vv_.cpu().numpy()))
    penalty = 50.0 * max(0.0, abs(cov - 0.95) - 0.02)   # tolera 0.93–0.97
    return vf + penalty

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50)
print("best:", study.best_params, "| val:", round(study.best_value, 4))

best: {'input_len': 210, 'kernel': 43, 'lr': 0.0038486956719697134} | val: -1.3554


In [8]:
print("trials concluídos:", len(study.trials))
df_tr = study.trials_dataframe()[["number","value","params_input_len","params_kernel","params_lr","state"]]
print(df_tr.sort_values("value").head(10).to_string(index=False))

trials concluídos: 50
 number     value  params_input_len  params_kernel  params_lr    state
     49 -1.355403               210             43   0.003849 COMPLETE
     24 -1.354269                60             45   0.003560 COMPLETE
     37 -1.323282                60             47   0.004337 COMPLETE
     22 -1.322334                60             45   0.003086 COMPLETE
     40 -1.313360               180             39   0.004346 COMPLETE
     39 -1.299177               180             43   0.004052 COMPLETE
     31 -1.290222                60             45   0.003552 COMPLETE
     21 -1.281760                60             47   0.002776 COMPLETE
      2 -1.278870                60             45   0.001050 COMPLETE
     30 -1.273342                60             37   0.002028 COMPLETE


## 7. Full multi-seed training

In [9]:
bp = study.best_params
INPUT_LEN = bp["input_len"]                      # sobrescreve o default
w = fz.make_windows(d, input_len=INPUT_LEN, h=H) # regenera janelas no tamanho tunado

runs = []
for s in [42, 43, 44]:
    torch.manual_seed(s); np.random.seed(s)
    mdl = fz.FZ0DLinear(INPUT_LEN, n_cov=w["n_cov"], kernel=bp["kernel"])
    mdl, vf = fz.fit(mdl, w, epochs=400, warmup=30, patience=40, lr=bp["lr"],
                     device=DEVICE, verbose=False)
    runs.append((s, vf, mdl)); print(f"seed {s}: val_FZ0={vf:.4f}")
runs.sort(key=lambda r: r[1])
seed_med, val_med, model = runs[len(runs)//2]
print("median seed:", seed_med, "| val_FZ0:", round(val_med, 4))

seed 42: val_FZ0=-1.3554
seed 43: val_FZ0=-1.2143
seed 44: val_FZ0=-1.2401
median seed: 44 | val_FZ0: -1.2401


## 8. Test backtest and comparison vs. the two-stage pipeline

In [10]:
# end-to-end (VaR, ES) in price -> loss space, aligned to test index
m, v, e = fz.predict(model, w, device=DEVICE)
yhat = to_price(m); q95 = to_price(m + v); es = to_price(m + e)
y = np.array([sra[t] for t in w["test_time"]])
L_e = np.log(y / yhat)
V_e = np.log(q95 / yhat)
E_e = np.maximum(np.log(es / yhat), V_e)          # ES >= VaR after transport

hits = (L_e > V_e).astype(int)
fz0_e2e = float(np.nanmean(vv.fz0_loss(L_e, np.maximum(V_e,1e-4), E_e, vv.A)))
print("=== FZ0-DLinear (end-to-end), Germany ===")
print(f"FZ0={fz0_e2e:.4f} | coverage={1-hits.mean():.3f} | Kupiec={vv.kupiec(hits,vv.A):.3f} "
      f"| DQ_str={vv.dq_stride(hits, V_e, H):.3f} | exc={int(hits.sum())}")

# baseline: best two-stage engine FZ0 for GER (from the risk-layer results)
# TCN via GJR-GARCH-t was the GER champion; recompute its FZ0 series for the DM test.
RF = f"result/v2p_result_steel_{CC.lower()}.xlsx"
_, L_b, _ = vv.load_losses(RF, CC, H, "tcn")
V_b, E_b = vv.rolling_var_garch(L_b, vv.A, "GJR")
W = vv.WINDOW
fz_b = vv.fz0_loss(L_b[W:], np.maximum(V_b[W:],1e-4), np.maximum(E_b[W:],V_b[W:]), vv.A)
fz_e = vv.fz0_loss(L_e, np.maximum(V_e,1e-4), E_e, vv.A)
n = min(len(fz_b), len(fz_e))
stat, p = vv.dm_loss(fz_e[:n], fz_b[:n], nw_lag=H)   # p<0.05 => end-to-end better
print(f"\nDM  end-to-end vs TCN/GJR-GARCH (2-stage): dFZ0={np.nanmean(fz_e[:n])-np.nanmean(fz_b[:n]):+.4f} "
      f"| p={p:.3f}  ({'end-to-end better' if p<0.05 else 'not significant'})")

=== FZ0-DLinear (end-to-end), Germany ===
FZ0=-3.1939 | coverage=0.885 | Kupiec=0.000 | DQ_str=0.000 | exc=115

DM  end-to-end vs TCN/GJR-GARCH (2-stage): dFZ0=+1.5791 | p=1.000  (not significant)


## 9. Camada conformal (ACI)

In [11]:
# --- ACI recalibration of the tuned e2e VaR (the framework's conformal stage) ---
W = vv.WINDOW
V_aci, E_aci = vv.aci_qr(L_e, V_e)                    # adaptive conformal on top
mask = np.isfinite(V_aci[W:])
La, Va = L_e[W:][mask], np.maximum(V_aci[W:][mask], 1e-4)
Ea = np.maximum(E_aci[W:][mask], Va)
hits_a = (La > Va).astype(int)
fz0_aci = float(np.nanmean(vv.fz0_loss(La, Va, Ea, vv.A)))
print("=== FZ0-DLinear + ACI, Germany ===")
print(f"FZ0={fz0_aci:.4f} | coverage={1-hits_a.mean():.3f} | "
      f"Kupiec={vv.kupiec(hits_a, vv.A):.3f} | DQ_str={vv.dq_stride(hits_a, Va, H):.3f} "
      f"| exc={int(hits_a.sum())}")

# DM vs the two-stage champion, on the SAME window
fz_e_aci = vv.fz0_loss(La, Va, Ea, vv.A)
n2 = min(len(fz_b), len(fz_e_aci))
stat2, p2 = vv.dm_loss(fz_e_aci[:n2], fz_b[:n2], nw_lag=H)
print(f"DM  e2e+ACI vs TCN/GJR-GARCH: dFZ0={np.nanmean(fz_e_aci[:n2])-np.nanmean(fz_b[:n2]):+.4f} "
      f"| p={p2:.3f}  ({'e2e better' if p2<0.05 else 'not significant'})")

=== FZ0-DLinear + ACI, Germany ===
FZ0=-4.1252 | coverage=0.953 | Kupiec=0.672 | DQ_str=0.733 | exc=35
DM  e2e+ACI vs TCN/GJR-GARCH: dFZ0=+0.4264 | p=0.999  (not significant)


In [12]:
# --- diagnostics: where do V_aci / E_aci live? ---
import numpy as np
W = vv.WINDOW
mask = np.isfinite(V_aci[W:])
La, Va = L_e[W:][mask], V_aci[W:][mask]
Ea = E_aci[W:][mask]
print("V_aci percentis:", np.round(np.percentile(Va, [1,5,25,50,75,95,99]), 5))
print("E_aci percentis:", np.round(np.percentile(Ea, [1,5,25,50,75,95,99]), 5))
print("dias com V_aci <= 1e-3:", int((Va <= 1e-3).sum()), "de", len(Va))
print("dias com E_aci <= V_aci + 1e-9 (cauda vazia, E colapsado em V):",
      int((Ea <= Va + 1e-9).sum()))
# quando ocorrem as 11 exceções? (concentradas no início = ACI ainda adaptando)
idx_exc = np.where(La > np.maximum(Va, 1e-4))[0]
print("posições das exceções (0 =", "início do teste):", idx_exc)
# decomposição do FZ0: quanto vem do ln(e)?
Ea_f = np.maximum(Ea, np.maximum(Va, 1e-4))
print("média ln(e):", round(float(np.mean(np.log(Ea_f))), 3),
      "| média v/e:", round(float(np.mean(np.maximum(Va,1e-4)/Ea_f)), 3))

V_aci percentis: [-0.00279  0.00156  0.00729  0.01139  0.01633  0.02953  0.03879]
E_aci percentis: [0.00331 0.00817 0.01223 0.0156  0.02063 0.03149 0.03879]
dias com V_aci <= 1e-3: 26 de 750
dias com E_aci <= V_aci + 1e-9 (cauda vazia, E colapsado em V): 58
posições das exceções (0 = início do teste): [  6   7  54  55  56 102 103 104 183 184 185 212 288 292 294 295 296 308
 344 367 406 497 505 506 507 508 509 563 564 565 628 629 630 660 664]
média ln(e): -4.16 | média v/e: 0.665
